# Download an OCR example dataset

Use the [energy-meter dataset](https://huggingface.co/datasets/henrik-dra/energy-meter) to try this repository on real images. It contains meter readings, so its scores do not measure license-plate or price-tag accuracy. The dataset files are about 361 MB.

Install the downloader dependencies in the notebook kernel with `%pip install huggingface_hub pyarrow Pillow`.

In [ ]:
from pathlib import Path

DOWNLOAD = False
OUTPUT = Path.cwd() / 'energy-meter-example'
DATASET = 'henrik-dra/energy-meter'
REVISION = 'b21598cf9109cdfb68e3f16276b78ce3a5d55639'


Set `DOWNLOAD = True` and run both code cells. The converter keeps train and test separate, saving test as `val`. Labels are written verbatim as UTF-8. Point the training configuration at the printed output directory. Use the same checkpoint and images to compare native and optimized inference.

In [ ]:
if DOWNLOAD:
    from io import BytesIO
    from tempfile import TemporaryDirectory
    from huggingface_hub import hf_hub_download
    from PIL import Image
    import pyarrow.parquet as pq

    if OUTPUT.exists():
        raise FileExistsError(f'Choose a new OUTPUT directory: {OUTPUT}')
    with TemporaryDirectory(dir=OUTPUT.parent) as staging:
        staging = Path(staging)
        dataset = staging / 'dataset'
        counts = {}
        for source, dest in [('train', 'train'), ('test', 'val')]:
            parquet = hf_hub_download(DATASET, f'data/{source}-00000-of-00001.parquet',
                                      repo_type='dataset', revision=REVISION)
            target = dataset / dest
            target.mkdir(parents=True)
            count = 0
            for batch in pq.ParquetFile(parquet).iter_batches(columns=['image', 'label'], batch_size=4):
                for row in batch.to_pylist():
                    stem = target / f'{count:05d}'
                    with Image.open(BytesIO(row['image']['bytes'])) as image:
                        image.convert('RGB').save(stem.with_suffix('.png'))
                    stem.with_suffix('.txt').write_text(row['label'], encoding='utf-8')
                    count += 1
            counts[dest] = count
        dataset.rename(OUTPUT)
    print(OUTPUT.resolve(), counts)
else:
    print('Set DOWNLOAD = True to fetch and convert the example.')
